# 지도 시각화

In [2]:
#!pip install folium

In [3]:
import folium

m = folium.Map(
    location=[37.566826, 126.9786567],
    zoom_start=11
)

m.save('data/map_basic.html')
print('지도 저장 완료: ../data/map_basic.html')
m

지도 저장 완료: ../data/map_basic.html


In [4]:
locations = [
    {'name': '경복궁', 'lat': 37.5796, 'lon': 126.9770, 'desc': '조선의 법궁'},
    {'name': '남산타워', 'lat': 37.5512, 'lon': 126.9882, 'desc': 'N서울타워'},
    {'name': '인사동', 'lat': 37.5741, 'lon': 126.9862, 'desc': '전통문화거리'},
]

m2 = folium.Map(location=[37.566, 126.977], zoom_start=13)

for loc in locations:
    folium.Marker(
        location=[loc['lat'], loc['lon']],
        popup=folium.Popup(f"<b>{loc['name']}</b><br>{loc['desc']}", max_width=200),
        tooltip=loc['name'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m2)

m2.save('data/map_markers.html')
m2

In [ ]:
import pandas as pd

data = [
    ('강남구', 37.5172, 127.0473, 564000),
    ('서초구', 37.4837, 127.0324, 444000),
    ('종로구', 37.5730, 126.9794, 153000),
    ('송파구', 37.5145, 127.1058, 671000),
    ('강서구', 37.5509, 126.8495, 588000),
    ('노원구', 37.6541, 127.0568, 514000),
]
df = pd.DataFrame(data, columns=['구', '위도', '경도', '인구'])

m3 = folium.Map(location=[37.566, 126.978], zoom_start=11)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=row['인구'] / 20000,
        color='crimson',
        fill=True,
        fill_color='crimson',
        fill_opacity=0.5,
        tooltip=f"{row['구']}: {row['인구']:,}명"
    ).add_to(m3)

m3.save('data/map_circle.html')
m3

### 따릉이 대여소 위치 표시

In [18]:
import pandas as pd
df = pd.read_excel('data/공공자전거 대여소 정보(25.12월 기준).xlsx',header= None)
df.head(10)
df_bike = df.iloc[5:, [1, 2, 4, 5, 8]]
df_bike.head()

,1,2,4,5,8
5,망원역 1번출구 앞,마포구,37.555649,126.910629,15
6,망원역 2번출구 앞,마포구,37.554951,126.910835,14
7,합정역 1번출구 앞,마포구,37.550629,126.914986,13
8,합정역 5번출구 앞,마포구,37.550007,126.914825,5
9,합정역 7번출구 앞,마포구,37.548645,126.912827,12


In [ ]:
# 컬럼명 수정
df_bike.columns = ['대여소명', '자치구', '위도', '경도', '거치대수']
df_bike.head()

In [ ]:
# 자치구로 그룹핑
st_top1 = df_bike.groupby('자치구')['거치대수'].sum().sort_values(ascending=False).head(1).reset_index()
st_top1

In [ ]:
st_top1 = st_top1.merge(df_bike, on='자치구')
st_top1.info()
st_top1['위도']  = st_top1['위도'].astype(float)
# st_top1['경도']  = st_top1['경도'].astype(float)
# st_top1.info()

In [ ]:
# 1. 거치대수 1등 자치구의 모든 대여소 추출
st_top1_area = df_bike.groupby('자치구')['거치대수'].sum().sort_values(ascending=False).head(1).reset_index()
st_top1 = st_top1_area.merge(df_bike, on='자치구')

# 2. 지도의 중심점을 잡기 위해 해당 자치구 대여소들의 평균 위도/경도 계산
center_lat = st_top1['위도'].mean()
center_lng = st_top1['경도'].mean()

# 3. 계산한 중심점으로 기본 지도 틀(m4) 만들기
# (그 자치구만 쏙 골라보기 위해 zoom_start를 13으로 가깝게 잡았습니다)
m4 = folium.Map(location=[center_lat, center_lng], zoom_start=13)

# 4. 반복문(for)을 돌며 1등 자치구 내 모든 대여소 위치 찍기
for _, row in st_top1.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        # 원이 너무 작거나 커지지 않게 기본 크기 7에 거치대 비례 값을 더해줍니다
        radius=7 + (row['거치대수_y'] / 5 if '거치대수_y' in row else row['거치대수'] / 5),
        color='darkgreen',      # 따릉이 상징색인 초록색 계열로 매칭
        fill=True,
        fill_color='lightgreen',
        fill_opacity=0.6,
        # 툴팁에 대여소 이름과 거치대수 표현
        tooltip=f"<b>대여소:</b> {row['대여소명']}<br><b>거치대수:</b> {row['거치대수_y'] if '거치대수_y' in row else row['거치대수']:,}대"
    ).add_to(m4)

# 5. 지도 저장 및 화면에 출력
m4.save('data/map_st_top1_all.html')
m4

In [45]:
# 마포구에대한 따릉이 대여소

# 1. 전체 데이터(df_bike)에서 '마포구' 데이터만 쏙 골라내기
df_mapo = df_bike[df_bike['자치구'] == '마포구']

# 2. 지도의 첫 중심점을 잡기 위해 마포구 대여소들의 평균 위도/경도 계산
center_lat = df_mapo['위도'].mean()
center_lng = df_mapo['경도'].mean()

# 3. 계산한 마포구 중심점으로 기본 지도 틀(m_mapo) 만들기
m_mapo = folium.Map(location=[center_lat, center_lng], zoom_start=13)

# 4. 반복문(for)을 돌며 마포구 내 모든 대여소 위치 찍기
for _, row in df_mapo.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        # 거치대수에 비례하여 원 크기 조절 (기본 크기 6 보장)
        radius=6 + (row['거치대수'] / 5),
        color='darkgreen',        # 테두리는 진한 초록색
        fill=True,
        fill_color='limegreen',  # 안쪽은 밝은 초록색
        fill_opacity=0.6,
        # 마우스 올렸을 때 대여소 정보 노출
        tooltip=f"<b>자치구:</b> 마포구<br><b>대여소:</b> {row['대여소명']}<br><b>거치대수:</b> {row['거치대수']:,}대"
    ).add_to(m_mapo)

# 5. 지도 파일 저장 및 화면 출력
m_mapo.save('data/map_mapo_bike.html')
m_mapo
